# CNN Classifier Training
**ECG Arrhythmia Detection | MIT-BIH | 8-Class | Phase 4**

### 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 2. Verify GPU

In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device} | GPU: {torch.cuda.get_device_name(0) if device.type == "cuda" else "NOT FOUND — Runtime > Change runtime type > T4 GPU"}')

### 3. Install & Import

In [ ]:
!pip install scikit-learn seaborn --quiet

import sys, json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.cuda.amp import autocast, GradScaler
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc
)
from sklearn.preprocessing import label_binarize

sys.path.append('/content/drive/MyDrive/ecg_project/models')
from cnn import ECGCNNClassifier, CLASS_NAMES, NUM_CLASSES, BEAT_LENGTH

### 4. Paths & Config

In [ ]:
DATA_DIR  = Path('/content/drive/MyDrive/ecg_project/processed_dataset')
CKPT_BASE = Path('/content/drive/MyDrive/ecg_project/checkpoints/cnn_classifier')
BEST_DIR  = CKPT_BASE / 'best'
EPOCH_DIR = CKPT_BASE / 'epochs'
PLOT_DIR  = CKPT_BASE / 'plots'

BATCH_SIZE = 128
EPOCHS     = 60
LR         = 1e-3
PATIENCE   = 10
CKPT_EVERY = 5

SHORT_NAMES = ['N', 'L', 'R', 'A', 'V', '/', 'E', 'F']

### 5. Class Distribution (Before Training)

In [ ]:
train_labels = np.load(DATA_DIR / 'train' / 'labels.npy')
class_counts = np.bincount(train_labels, minlength=NUM_CLASSES).astype(np.float32)

plt.figure(figsize=(10, 4))
bars = plt.bar(SHORT_NAMES, class_counts, color='steelblue', edgecolor='black')
for bar, count in zip(bars, class_counts):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
             f'{int(count)}', ha='center', va='bottom', fontsize=9)
plt.title('Class Distribution — Training Set (MIT-BIH)')
plt.xlabel('Beat Class'); plt.ylabel('Sample Count')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'class_distribution.png', dpi=150)
plt.show()
print('Class imbalance visible → class weights applied in loss function')

### 6. Dataset & DataLoader

In [ ]:
class ECGDataset(Dataset):
    def __init__(self, split):
        self.beats  = torch.tensor(np.load(DATA_DIR / split / 'beats.npy'),  dtype=torch.float32).unsqueeze(1)
        self.labels = torch.tensor(np.load(DATA_DIR / split / 'labels.npy'), dtype=torch.long)
    def __len__(self):          return len(self.labels)
    def __getitem__(self, idx): return self.beats[idx], self.labels[idx]

train_loader = DataLoader(ECGDataset('train'), batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(ECGDataset('val'),   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(ECGDataset('test'),  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {len(train_loader.dataset)} | Val: {len(val_loader.dataset)} | Test: {len(test_loader.dataset)}')

### 7. Class Weights (Fix Imbalance)

In [ ]:
class_weights = torch.tensor(1.0 / (class_counts + 1e-6)).to(device)
class_weights = class_weights / class_weights.sum() * NUM_CLASSES

for i, (cnt, name) in enumerate(zip(class_counts, CLASS_NAMES.values())):
    print(f'  Class {i} | {int(cnt):>6} samples | weight: {class_weights[i]:.4f} | {name}')

### 8. Model, Loss & Optimizer

In [ ]:
model     = ECGCNNClassifier(num_classes=NUM_CLASSES, dropout_rate=0.4).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5, verbose=True)
scaler    = GradScaler()

model.model_summary()

### 9. Training Loop

In [ ]:
best_val_loss = float('inf')
patience_ctr  = 0
history       = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'lr': []}

for epoch in range(1, EPOCHS + 1):

    model.train()
    train_loss = 0.0
    for beats, labels in train_loader:
        beats, labels = beats.to(device), labels.to(device)
        optimizer.zero_grad()
        with autocast():
            loss = criterion(model(beats), labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    model.eval()
    val_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for beats, labels in val_loader:
            beats, labels = beats.to(device), labels.to(device)
            logits    = model(beats)
            val_loss += criterion(logits, labels).item()
            correct  += (logits.argmax(1) == labels).sum().item()
            total    += labels.size(0)
    val_loss /= len(val_loader)
    val_acc   = correct / total * 100

    current_lr = optimizer.param_groups[0]['lr']
    scheduler.step(val_loss)
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['lr'].append(current_lr)

    print(f'Epoch {epoch:03d} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | Acc: {val_acc:.2f}% | LR: {current_lr:.6f}')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_ctr  = 0
        torch.save(model.state_dict(), BEST_DIR / 'cnn_best.pth')
        print(f'  → Best model saved (val_loss: {best_val_loss:.4f})')
    else:
        patience_ctr += 1

    if epoch % CKPT_EVERY == 0:
        torch.save(model.state_dict(), EPOCH_DIR / f'cnn_epoch_{epoch}.pth')
        print(f'  → Checkpoint saved (epoch {epoch})')

    if patience_ctr >= PATIENCE:
        print(f'Early stopping at epoch {epoch}')
        break

print(f'\nTraining complete | Best Val Loss: {best_val_loss:.4f}')

### 10. Training Curves (Loss & Accuracy)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history['train_loss'], label='Train Loss')
ax1.plot(history['val_loss'],   label='Val Loss')
ax1.set_title('Loss Curve'); ax1.set_xlabel('Epoch'); ax1.legend()
ax2.plot(history['val_acc'], color='green', label='Val Accuracy')
ax2.set_title('Validation Accuracy'); ax2.set_xlabel('Epoch'); ax2.legend()
plt.tight_layout()
plt.savefig(PLOT_DIR / 'training_curves.png', dpi=150)
plt.show()

### 11. Learning Rate Schedule

In [ ]:
plt.figure(figsize=(8, 3))
plt.plot(history['lr'], color='orange', marker='o', markersize=3)
plt.title('Learning Rate Schedule')
plt.xlabel('Epoch'); plt.ylabel('Learning Rate')
plt.yscale('log')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'lr_schedule.png', dpi=150)
plt.show()

### 12. Evaluate on Test Set

In [ ]:
model.load_state_dict(torch.load(BEST_DIR / 'cnn_best.pth'))
model.eval()

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for beats, labels in test_loader:
        logits = model(beats.to(device))
        probs  = torch.softmax(logits, dim=-1).cpu()
        preds  = logits.argmax(1).cpu()
        all_preds.extend(preds.numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs  = np.array(all_probs)

print(classification_report(all_labels, all_preds, target_names=list(CLASS_NAMES.values()), digits=4))

### 13. Confusion Matrix

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=SHORT_NAMES, yticklabels=SHORT_NAMES)
plt.title('Confusion Matrix — CNN Classifier')
plt.ylabel('True Label'); plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'confusion_matrix.png', dpi=150)
plt.show()

### 14. Per-Class Precision, Recall & F1

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

precision = precision_score(all_labels, all_preds, average=None, zero_division=0)
recall    = recall_score(all_labels,    all_preds, average=None, zero_division=0)
f1        = f1_score(all_labels,        all_preds, average=None, zero_division=0)

x = np.arange(NUM_CLASSES)
w = 0.25
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x - w, precision, w, label='Precision', color='steelblue')
ax.bar(x,     recall,    w, label='Recall',    color='darkorange')
ax.bar(x + w, f1,        w, label='F1 Score',  color='green')
ax.set_xticks(x); ax.set_xticklabels(SHORT_NAMES)
ax.set_ylim(0, 1.1); ax.set_ylabel('Score'); ax.set_xlabel('Beat Class')
ax.set_title('Per-Class Precision, Recall & F1 — CNN Classifier')
ax.legend(); ax.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1))
plt.tight_layout()
plt.savefig(PLOT_DIR / 'per_class_metrics.png', dpi=150)
plt.show()

### 15. ROC Curve & AUC (Per Class)

In [ ]:
labels_bin = label_binarize(all_labels, classes=list(range(NUM_CLASSES)))

plt.figure(figsize=(10, 7))
colors = plt.cm.tab10(np.linspace(0, 1, NUM_CLASSES))

for i, (name, color) in enumerate(zip(SHORT_NAMES, colors)):
    fpr, tpr, _ = roc_curve(labels_bin[:, i], all_probs[:, i])
    roc_auc     = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=color, label=f'Class {name} (AUC = {roc_auc:.4f})')

plt.plot([0, 1], [0, 1], 'k--', linewidth=0.8)
plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate')
plt.title('ROC Curve — CNN Classifier (One-vs-Rest)')
plt.legend(loc='lower right', fontsize=8)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'roc_auc.png', dpi=150)
plt.show()

### 16. Sample Beat Predictions

In [ ]:
test_beats  = np.load(DATA_DIR / 'test' / 'beats.npy')
test_lbls   = np.load(DATA_DIR / 'test' / 'labels.npy')

# Pick one sample per class
fig, axes = plt.subplots(2, 4, figsize=(16, 6))
axes = axes.flatten()

for class_idx in range(NUM_CLASSES):
    idx   = np.where(test_lbls == class_idx)[0][0]
    beat  = test_beats[idx]
    x_in  = torch.tensor(beat, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        pred_idx = model(x_in).argmax(1).item()
        conf     = torch.softmax(model(x_in), dim=-1).max().item()

    color = 'green' if pred_idx == class_idx else 'red'
    axes[class_idx].plot(beat, color=color, linewidth=0.8)
    axes[class_idx].set_title(
        f'True: {SHORT_NAMES[class_idx]} | Pred: {SHORT_NAMES[pred_idx]}\nConf: {conf:.2%}',
        fontsize=9, color=color
    )
    axes[class_idx].axis('off')

plt.suptitle('Sample Beat Predictions (Green=Correct, Red=Wrong)', fontsize=12)
plt.tight_layout()
plt.savefig(PLOT_DIR / 'sample_predictions.png', dpi=150)
plt.show()

### 17. Misclassification Analysis (Top Confused Pairs)

In [ ]:
wrong_mask    = all_preds != all_labels
wrong_true    = all_labels[wrong_mask]
wrong_pred    = all_preds[wrong_mask]

from collections import Counter
pairs   = Counter(zip(wrong_true, wrong_pred))
top10   = pairs.most_common(10)
labels_ = [f'{SHORT_NAMES[t]}→{SHORT_NAMES[p]}' for (t, p), _ in top10]
counts_ = [c for _, c in top10]

plt.figure(figsize=(10, 4))
plt.bar(labels_, counts_, color='tomato', edgecolor='black')
plt.title('Top 10 Misclassification Pairs (True → Predicted)')
plt.xlabel('Class Pair'); plt.ylabel('Count')
plt.tight_layout()
plt.savefig(PLOT_DIR / 'misclassification_analysis.png', dpi=150)
plt.show()

### 18. Save Labels JSON & Final Summary

In [ ]:
with open(BEST_DIR / 'class_labels.json', 'w') as f:
    json.dump({str(k): v for k, v in CLASS_NAMES.items()}, f, indent=2)

print('All files saved to Drive:')
print(f'  best/   → cnn_best.pth, class_labels.json')
print(f'  epochs/ → cnn_epoch_5.pth ... cnn_epoch_N.pth')
print(f'  plots/  → training_curves.png, confusion_matrix.png,')
print(f'            per_class_metrics.png, roc_auc.png,')
print(f'            sample_predictions.png, misclassification_analysis.png,')
print(f'            class_distribution.png, lr_schedule.png')